# 饲喂料量vs换料vs标准

## README

### 4栋2单元数据

In [15]:
# 读取数据
import pandas as pd
from feed_analysis.config.path import PATH_DATA, PATH_FEED_PROCESSED, PATH_FIGURE_HTML
from feed_analysis.config.coding_schema import STD_HEADER_NAME
from feed_analysis.feed_pipeline.utils.age import recalibrate_age

# # 更新日龄
# recalibrate_age(PATH_FEED_PROCESSED/'育肥4-2_column.parquet', reference_date='2025-8-25', reference_age=23)         # 根据杨乐乐主管1.4提供信息
# recalibrate_age(PATH_FEED_PROCESSED/'育肥4-2_build.parquet', reference_date='2025-8-25', reference_age=23)         # 根据杨乐乐主管1.4提供信息

# 读取喂食量数据 
df_42_col = pd.read_parquet(PATH_FEED_PROCESSED/'育肥4-2_column.parquet', engine='pyarrow')
df_42_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥4-2_build.parquet', engine='pyarrow')

# 读取存栏量数据
df_42_num = pd.read_excel(PATH_DATA / 'ori' / '育肥4-2单元存栏数.xlsx', header=3).rename(columns=STD_HEADER_NAME).sort_values(by=['Date'])
df_42_num['Date'] = pd.to_datetime(df_42_num['Date']).dt.date

# 计算头均值
df_42_build['stock_num'] = df_42_build['Date'].map(df_42_num.set_index('Date')['stock_num'])
df_42_build.dropna(subset=['stock_num'], inplace=True)
df_42_build['avg_food_kg'] = df_42_build['food_total_kg'] / df_42_build['stock_num']

# 插补缺失值
from feed_analysis.feed_pipeline.utils.interpolate import interpolation
df_42_build = interpolation(df_42_build, index_col='age', )

总趋势变化

In [16]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'avg_food_kg', '育肥4-2单元料头均值', color='red')

### log-log 回归（best model）

In [17]:
from feed_analysis.growing_fit.log_log_fit import poly_log_regression_with_smearing
df_42_build, model_fit = poly_log_regression_with_smearing(df_42_build, x_col="age", y_col="avg_food_kg", degree=2, alpha=0.15, show_metrics=False)  # R2 0.94
from feed_analysis.growing_fit.log_log_fit import plot_true_pred

fig = plot_true_pred(df_42_build, x_name='age', plot_title='四栋喂食量时序图')
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   
fig.write_html(PATH_FIGURE_HTML/'四栋喂食量时序图.html')
fig.show()

可能因素：断奶、饲料


### 3栋数据

三栋总体数据&存栏量

In [18]:
# 数据载入
import pandas as pd
from feed_analysis.config.path import PATH_DATA, PATH_FEED_PROCESSED
from feed_analysis.config.coding_schema import STD_HEADER_NAME
from feed_analysis.feed_pipeline.utils.age import recalibrate_age

# 读取喂食量数据 
df_31_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-1_build.parquet', engine='pyarrow')
df_32_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-2_build.parquet', engine='pyarrow')
df_33_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-3_build.parquet', engine='pyarrow')
df_34_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-4_build.parquet', engine='pyarrow')
# 读取存栏量数据
df_3_num = pd.read_excel(PATH_DATA / 'ori' / '育肥3栋总存栏数据.xlsx').rename(columns=STD_HEADER_NAME).sort_values(by=['Date'], ascending=True)
df_3_num['Date'] = df_3_num['Date']

# 计算头均值
df_3_build = pd.concat([df_31_build['food_total_kg'], df_32_build['food_total_kg'], df_33_build['food_total_kg'], df_34_build['food_total_kg']], axis=1)
df_3_build['total'] = df_3_build.sum(axis=1)
df_3_build.columns = ['3_1_feed', '3_2_feed', '3_3_feed', '3_4_feed', 'food_total_sum']

df_3_build['Date'] = df_31_build['Date']
df_3_build['age'] = df_31_build['age']

df_3_build['stock_num'] = df_3_build['Date'].map(df_3_num.set_index('Date')['stock_num'])
df_3_build['avg_food_kg'] = df_3_build['food_total_sum'] / df_3_build['stock_num']

# # 筛选日期
df_3_build = df_3_build.loc[df_3_build['Date'].between(pd.to_datetime('2025-08-28').date(), pd.to_datetime('2025-12-30').date()), :] 

# 插补缺失值
from feed_analysis.feed_pipeline.utils.interpolate import interpolation
df_3_build = interpolation(df_3_build, index_col='age', )

In [19]:
from feed_analysis.growing_fit.log_log_fit import poly_log_regression_with_smearing
df_3_build, model_fit_3build = poly_log_regression_with_smearing(df_3_build, x_col="age", y_col="avg_food_kg", degree=2, alpha=0.15, show_metrics=False)        # R2 0.972
from feed_analysis.growing_fit.log_log_fit import plot_true_pred

fig = plot_true_pred(df_3_build,  x_name='age', y_true_name='avg_food_kg', y_pred_name='food_fit', plot_title='育肥3栋生长曲线拟合')
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green",annotation=dict(text='换料'))   
fig.write_html(PATH_FIGURE_HTML/'三栋喂食量时序图.html')
fig.show()

c:\Users\yixua\.conda\envs\env\Lib\site-packages\plotly\io\_json.py:558: UserWarning:

Discarding nonzero nanoseconds in conversion.



In [ ]:
import plotly.graph_objects as go
from feed_analysis.growing_fit.log_log_fit import poly_log_regression_with_smearing
df_3_build, model_fit_3build = poly_log_regression_with_smearing(df_3_build, x_col="age", y_col="avg_food_kg", degree=2, alpha=0.15, show_metrics=False)        # R2 0.972
from feed_analysis.growing_fit.log_log_fit import plot_true_pred

# 添加四栋的数据
df_3_build['compare_4_true'] = df_3_build['age'].map(df_42_build.set_index('age')['avg_food_kg'])
df_3_build['compare_4_fit'] = df_3_build['age'].map(df_42_build.set_index('age')['food_fit'])

# fig = plot_true_pred(df_3_build,  x_name='age', y_true_name='avg_food_kg', y_pred_name='food_fit', plot_title='育肥3栋生长曲线拟合')
# 添加3栋数据
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['avg_food_kg'], mode='markers', name='三栋真实值', opacity=1))
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['food_fit'], mode='lines', name='三栋拟合值', opacity=0.4))
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green",annotation=dict(text='换料'))  

# 添加4栋数据
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['compare_4_true'], mode='markers', name='四栋真实值', opacity=1))
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['compare_4_fit'], mode='lines', name='四栋拟合值', opacity=0.4))
fig.write_html(PATH_FIGURE_HTML / '三栋喂食量时序图.html')
fig.show()

### vs标准采食量

In [ ]:

import pandas as pd

# 读取标准采食量数据
df_std = pd.read_excel(PATH_DATA / 'ori' / '汇兴牧业-标准日龄饲料.xlsx', header=1)

df_3_build['std_feed'] = df_3_build['age'].map(df_std.set_index('日龄')['日喂料量(kg/头)'])

from feed_analysis.growing_fit.log_log_fit import plot_true_pred

fig = plot_true_pred(df_3_build,  x_name='age', y_true_name='avg_food_kg', y_pred_name='std_feed', plot_title='育肥3栋喂食vs2026汇兴牧业标准')
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green",annotation=dict(text='换料'))   
fig.write_html(PATH_FIGURE_HTML/'育肥3栋喂食vs标准.html')
fig.show()

c:\Users\yixua\.conda\envs\env\Lib\site-packages\plotly\io\_json.py:558: UserWarning:

Discarding nonzero nanoseconds in conversion.



In [23]:

import pandas as pd

# 读取标准采食量数据
df_std = pd.read_excel(PATH_DATA / 'ori' / '汇兴牧业-标准日龄饲料.xlsx', header=1)

df_42_build['std_feed'] = df_42_build['age'].map(df_std.set_index('日龄')['日喂料量(kg/头)'])

from feed_analysis.growing_fit.log_log_fit import plot_true_pred
fig = plot_true_pred(df_42_build,  x_name='age', y_true_name='avg_food_kg', y_pred_name='std_feed', plot_title='育肥4栋2单元喂食vs2026汇兴牧业标准')
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green",annotation=dict(text='换料'))   
fig.write_html(PATH_FIGURE_HTML/'育肥4栋2单元喂食vs2026汇兴牧业标准.html')
fig.show()

### 结论

**基于标准对比**
- 目前在不限量的情况下，当前2026牧原标准低于猪只日饲料量
- 推荐提高喂食标准，以促进猪只生长

**三栋四栋对比**
- 70天的换料会有一个明显的进食料降低
- 四栋转舍日龄23，三栋转舍日龄32，而四栋在转舍7天之后出现了明显的下降趋势，结合[《母猪场决定小猪命运》](https://mp.weixin.qq.com/s/06qmSumNojJQ5b8CyW3Ltg)文章，可能是**断奶过早导致的免疫力降低**，在七天时处于一个“免疫力底下”时期。

**未来分析路径**
- 考虑使用拟合值 + 给定饲料的料肉比，给出日增重曲线
    